In [ ]:
import time
from queue import Queue
from threading import Thread

try:
    from tqdm.auto import tqdm
    from tqdm.contrib.concurrent import thread_map
except Exception:
    tqdm = None


_Q = Queue()
_WORKERS = []
_STOP = object()


def start_workers(n=4):
    if _WORKERS:
        return

    def worker():
        while True:
            job = _Q.get()
            try:
                if job is _STOP:
                    return
                fn, args, kwargs = job
                fn(*args, **kwargs)
            finally:
                _Q.task_done()

    for _ in range(n):
        t = Thread(target=worker, daemon=True)
        t.start()
        _WORKERS.append(t)


def submit_job(fn, /, *args, **kwargs):
    if not _WORKERS:
        start_workers()
    _Q.put((fn, args, kwargs))


def shutdown_workers(progress=False, desc="Finishing...", poll_s=0.01):
    total = _Q.unfinished_tasks
    pbar = tqdm(total=total, desc=desc, unit="jobs") if (progress and tqdm and total) else None
    try:
        while pbar and _Q.unfinished_tasks:
            pbar.n = total - _Q.unfinished_tasks
            pbar.refresh()
            time.sleep(poll_s)
        pbar.n = total - _Q.unfinished_tasks
        pbar.refresh()
        _Q.join()
        pbar.n = total - _Q.unfinished_tasks
        pbar.refresh()
    finally:
        if pbar:
            pbar.close()

    for _ in _WORKERS:
        _Q.put(_STOP)
    _Q.join()
    for t in _WORKERS:
        t.join()
    _WORKERS.clear()

In [ ]:
from ultralytics.engine.results import Boxes, Results
from ultralytics.utils.nms import TorchNMS
from ultralytics.utils.plotting import save_one_box
import numpy as np
import os
from PIL import Image
import requests
import torch
from glob import glob
from pathlib import Path

def read_image(url : str):
    return Image.open(requests.get(url, stream=True).raw)

def get_boxes(path, w, h):
    d = np.loadtxt(path)
    if d.ndim == 1: d = d[None]
    # Input: class, x, y, w, h, conf
    x, y, bw, bh = d[:, 1] * w, d[:, 2] * h, d[:, 3] * w, d[:, 4] * h
    # Output: x1, y1, x2, y2, conf, class
    return np.column_stack((x - bw/2, y - bh/2, x + bw/2, y + bh/2, d[:, 5], d[:, 0]))

def save_crops(results : Results, dst : str):
    if os.path.exists(dst) and not os.path.isdir(dst):
        raise NotADirectoryError(f'Crop destination: "{dst}" is not a valid directory.')
    os.makedirs(dst, exist_ok=True)
    
    for i, data in enumerate(results.summary()):
        x1, y1, x2, y2 = [data["box"][k] for k in ["x1", "y1", "x2", "y2"]]
        box = torch.tensor([x1,y1,x2,y2])
        conf = data["confidence"] * 100
        name = "crop{}_conf_{:.0f}_box_{:.0f}_{:.0f}_{:.0f}_{:.0f}_end.jpg".format(i, conf, *box.round().clamp_min(0).long().tolist())
        submit_job(
            save_one_box,
            xyxy=box,
            im=results.orig_img[..., ::-1],
            file=Path(os.path.join(dst, name)),
            BGR=True
        )

SHARELINK = "https://anon.erda.au.dk/share_redirect/H4AVdnrmtT/2017/Narsarsuaq/"
RESULTS = "/home/asger/Repositories/jet_detect/results/Narsarsuaq_2017"

def visualize(
        name : str, 
        output : str | None=None,
        crop_dir : str | None=None,
        dim : tuple[int, int]=(6080, 3420), 
        scale : float=0.125,
        iou_threshold : float | None=None,
        conf_threshold : float | None=None,
        **kwargs
    ):
    detections = Boxes(get_boxes(os.path.join(RESULTS, name + ".txt"), *dim), dim)
    url = SHARELINK + name + ".JPG"
    image = read_image(url)

    if conf_threshold is not None:
        detections = detections[detections.conf >= conf_threshold]
    if iou_threshold is not None:
        keep = TorchNMS.fast_nms(torch.from_numpy(detections.xyxy), torch.from_numpy(detections.conf), iou_threshold)
        detections = detections[keep]

    result = Results(np.asarray(image), path=url, names={0 : ""}, boxes=torch.from_numpy(detections.data))    

    if crop_dir is not None:
        save_crops(results=result, dst=crop_dir)

    if output is None:
        return # plot

    plot = Image.fromarray(result.plot(**kwargs)).resize((round(d * scale) for d in dim))
    
    if not os.path.exists(os.path.dirname(output)):
        raise FileNotFoundError(f'Directory of output: {output}, does not exist.')
    plot.save(output)

all_names = sorted([os.path.splitext(os.path.relpath(path, RESULTS))[0] for path in glob(RESULTS + "/*/*")])

def proc(name : str):
    dst = os.path.join(RESULTS, "visualizations", f'{name}_pred.jpg')
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    visualize(
        name=name, 
        output=dst,
        crop_dir=os.path.join(RESULTS, "crops", name),  
        conf_threshold=0.5, 
        font_size=72
    )
    
thread_map(proc, all_names, desc="Processing predictions...", tqdm_class=tqdm, max_workers=15)
shutdown_workers(progress=True)

In [ ]:
from pyremotedata.implicit_mount import IOHandler

with IOHandler() as io:
    io.cd("data/dryas")
    io.execute_command("mkdir -p crops")
    io.cd("crops")
    io.sync("/home/asger/Repositories/jet_detect/results/Narsarsuaq_2017/crops", direction="up", progress=True)
    print(io.ls())